# 8. Envío Final — Predicting Smartphone Addiction

El episodio produjo **ocho archivos de submission** a lo largo de seis notebooks de modelado. Este
cierra el trabajo: decide **cuál se envía y por qué**, lo verifica, lo manda, y deja registrado qué
seleccionar para el scoring privado.

## La decisión

Se envía **`6_xgboost_optuna_submission.csv`**: el XGBoost con hiperparámetros optimizados, sin
ensamblar.

**Por qué ese y no el ensamble**, que tenía mejor AUC out-of-fold (0,96519 contra 0,96507):

> Los pesos del ensamble se eligieron **maximizando el AUC sobre el mismo OOF con el que después se
> midió la ganancia**. Esa ganancia está inflada por construcción — se eligió el máximo de un conjunto
> de opciones ruidosas — y no es lo que se puede esperar fuera de muestra.
>
> El leaderboard ya lo demostró una vez: el blend del notebook 4 tenía +0,00016 en OOF sobre XGBoost
> solo, y sacó **exactamente el mismo score público, 0,96634**. La ganancia era sesgo, no mejora.

La ventaja del modelo tuneado, en cambio, es genuina. Sus hiperparámetros se eligieron sobre una
submuestra de 250k filas con 3 folds, y se evaluó sobre el OOF completo de 5 folds — la selección no
vio el mismo esquema de evaluación. Y `7_ruido_y_significancia.ipynb` la verificó por tres vías
independientes:

| Evidencia | Resultado |
|---|---|
| Bootstrap pareado (200 resamples) | z = 11,1 · ganó en **200/200** |
| Comparación por fold | ganó en **5/5** (t pareado 8,33, p = 0,0011) |
| Reentrenamiento con otras semillas | ganó en **3/3** (desvío entre semillas 10x menor que la ventaja) |

**Dataset:** Playground Series S6E8 · **Métrica:** ROC AUC

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import os
import warnings
warnings.filterwarnings("ignore")

COMPETITION  = "playground-series-s6e8"
DIR_SUB      = "../submissions"
A_ENVIAR     = "6_xgboost_optuna_submission.csv"

## 1. Inventario de lo Producido

Nada de esta tabla está transcrito a mano: cada columna se deriva de su fuente.

- **Los archivos** salen de listar `../submissions/`.
- **El CV** se **recalcula** a partir de los vectores out-of-fold que cada notebook guardó en
  `../models/*_oof.npy`. Si un notebook se vuelve a correr con otros parámetros, este número cambia
  solo.
- **El score público** se consulta a la **API de Kaggle**, que es la única fuente que lo conoce.

Los tres blends aparecen sin CV porque sus notebooks guardaron el OOF de los modelos individuales pero
no el de la mezcla. Es una omisión real de esos notebooks, y prefiero que quede visible antes que
taparla copiando el número a mano.

In [2]:
from sklearn.metrics import roc_auc_score

DIR_MODELS = "../models"

# --- CV: recalculado desde los OOF guardados ---
y = pd.read_csv("../data/train.csv")["addicted_label"].values
oofs = {f[:-len("_oof.npy")]: np.load(os.path.join(DIR_MODELS, f))
        for f in sorted(os.listdir(DIR_MODELS)) if f.endswith("_oof.npy")}
cv = {k: roc_auc_score(y, v) for k, v in oofs.items()}
print(f"CV recalculado desde {len(cv)} vectores OOF: {', '.join(cv)}")

CV recalculado desde 4 vectores OOF: 3_xgboost, 4_lightgbm, 5_red_neuronal, 6_xgboost_optuna


In [3]:
# --- Historial de envios: consultado a la API ---
try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    crudo = pd.DataFrame([{
        "archivo":    s.file_name,
        "fecha":      s.date,
        "estado":     str(s.status).rsplit(".", 1)[-1],
        "lb_publico": pd.to_numeric(s.public_score, errors="coerce"),
    } for s in api.competition_submissions(COMPETITION, page_size=100)])

    # un mismo archivo puede haberse enviado mas de una vez
    envios = (crudo.sort_values("fecha")
                   .groupby("archivo")
                   .agg(veces=("estado", "size"),
                        lb_publico=("lb_publico", "max"),
                        ultimo_estado=("estado", "last"))
                   .reset_index())
    API_OK = True
    print(f"API OK: {len(crudo)} envios historicos sobre {len(envios)} archivos distintos.")
except Exception as e:
    API_OK = False
    envios = pd.DataFrame(columns=["archivo", "veces", "lb_publico", "ultimo_estado"])
    print(f"No se pudo consultar la API ({type(e).__name__}).")
    print("  -> La tabla muestra solo la parte local; el estado de envio queda DESCONOCIDO,")
    print("     que no es lo mismo que 'sin enviar'.")

API OK: 6 envios historicos sobre 5 archivos distintos.


In [4]:
# --- Union de las tres fuentes ---
inv = pd.DataFrame({"archivo": sorted(f for f in os.listdir(DIR_SUB) if f.endswith(".csv"))})
inv["cv_auc"] = inv["archivo"].str.replace("_submission.csv", "", regex=False).map(cv)
inv = inv.merge(envios, on="archivo", how="left")
inv["enviado"] = inv["veces"].notna() if API_OK else "?"

print(inv.round(5).to_string(index=False))
print(f"\nArchivos generados: {len(inv)}")
if API_OK:
    sin_enviar = inv.loc[~inv["enviado"], "archivo"]
    print(f"Ya enviados: {int(inv['enviado'].sum())}  |  Sin enviar: {len(sin_enviar)}")
    print(f"\nSin enviar: {', '.join(sin_enviar)}")
else:
    print("Estado de envio: DESCONOCIDO (la API no respondio, ver celda anterior)")
assert A_ENVIAR in set(inv["archivo"]), f"{A_ENVIAR} no esta en ../submissions"

                        archivo  cv_auc  veces  lb_publico ultimo_estado  enviado
         2_logit_submission.csv     NaN    1.0     0.91909      COMPLETE     True
       3_xgboost_submission.csv 0.96478    1.0     0.96634      COMPLETE     True
4_lgbm_xgb_blend_submission.csv     NaN    1.0     0.96634      COMPLETE     True
      4_lightgbm_submission.csv 0.96422    NaN         NaN           NaN    False
    5_ensamble_3_submission.csv     NaN    NaN         NaN           NaN    False
  5_red_neuronal_submission.csv 0.94092    2.0     0.94270      COMPLETE     True
      6_ensamble_submission.csv     NaN    1.0     0.96654      COMPLETE     True
6_xgboost_optuna_submission.csv 0.96507    NaN         NaN           NaN    False

Archivos generados: 8
Ya enviados: 5  |  Sin enviar: 3

Sin enviar: 4_lightgbm_submission.csv, 5_ensamble_3_submission.csv, 6_xgboost_optuna_submission.csv


## 2. Qué NO se envía, y por qué

Antes de gastar un cupo diario conviene descartar lo redundante. Dos de los archivos sin enviar no
aportan nada:

In [5]:
def cargar(f):
    return pd.read_csv(os.path.join(DIR_SUB, f))["addicted_label"].values


# a) 5_ensamble_3 vs el blend del notebook 4
a = cargar("4_lgbm_xgb_blend_submission.csv")
b = cargar("5_ensamble_3_submission.csv")
print(f"5_ensamble_3 vs 4_lgbm_xgb_blend: identicos = {np.allclose(a, b)}")
print("  La busqueda de 3 vias le dio peso 0 a la red y aterrizo en los mismos pesos 0.30/0.70,")
print("  asi que el archivo es un duplicado del que ya saco 0.96634. Enviarlo no mide nada.\n")

# b) LightGBM solo: sabemos que es peor, y de forma significativa
print("4_lightgbm: CV 0.96422 contra 0.96478 de XGBoost.")
print("  El test pareado del notebook 7 le da z = 14.6 a esa diferencia: es peor de verdad,")
print("  no un empate. Solo valdria la pena enviarlo para completar la tabla.")

5_ensamble_3 vs 4_lgbm_xgb_blend: identicos = True
  La busqueda de 3 vias le dio peso 0 a la red y aterrizo en los mismos pesos 0.30/0.70,
  asi que el archivo es un duplicado del que ya saco 0.96634. Enviarlo no mide nada.

4_lightgbm: CV 0.96422 contra 0.96478 de XGBoost.
  El test pareado del notebook 7 le da z = 14.6 a esa diferencia: es peor de verdad,
  no un empate. Solo valdria la pena enviarlo para completar la tabla.


## 3. Verificación del Archivo a Enviar

Los mismos chequeos de formato que hace cada notebook de modelo, más una comprobación que acá importa
especialmente: que el archivo sea **realmente distinto** de los ya enviados. Si su correlación de
rangos con `3_xgboost` fuera 1,0, estaríamos gastando un cupo en el mismo modelo.

In [6]:
sub    = pd.read_csv(os.path.join(DIR_SUB, A_ENVIAR))
sample = pd.read_csv("../data/sample_submission.csv")

assert list(sub.columns) == list(sample.columns), "columnas no coinciden"
assert len(sub) == len(sample), "cantidad de filas no coincide"
assert (sub["id"].values == sample["id"].values).all(), "ids no alineados"
assert sub["addicted_label"].between(0, 1).all(), "valores fuera de [0, 1]"
assert sub["addicted_label"].notna().all(), "hay nulos"
print(f"Formato OK — {len(sub):,} filas")
print(f"  probabilidad media: {sub['addicted_label'].mean():.4f}  (tasa base en train: 0.7094)")

print("\nCorrelacion de rangos contra lo ya enviado:")
p_nuevo = sub["addicted_label"].values
for f in ["3_xgboost_submission.csv", "4_lgbm_xgb_blend_submission.csv",
          "6_ensamble_submission.csv"]:
    rho = spearmanr(p_nuevo, cargar(f)).statistic
    print(f"  vs {f:36s} rho = {rho:.6f}")
print("\n  Alta pero no 1.0: es un modelo distinto, aunque de la misma familia.")

Formato OK — 296,302 filas
  probabilidad media: 0.7095  (tasa base en train: 0.7094)

Correlacion de rangos contra lo ya enviado:
  vs 3_xgboost_submission.csv             rho = 0.999220
  vs 4_lgbm_xgb_blend_submission.csv      rho = 0.999083
  vs 6_ensamble_submission.csv            rho = 0.999828

  Alta pero no 1.0: es un modelo distinto, aunque de la misma familia.


## 4. Envío

In [7]:
MSG = "XGBoost + Optuna (25 trials) - CV AUC=0.96507 - verificado vs semillas y folds"

!kaggle competitions submit -c {COMPETITION} -f "{DIR_SUB}/{A_ENVIAR}" -m "{MSG}"

Successfully submitted to Predicting Smartphone Addiction



  0%|          | 0.00/5.26M [00:00<?, ?B/s]
  0%|          | 16.0k/5.26M [00:00<00:41, 131kB/s]
 10%|▉         | 528k/5.26M [00:00<00:06, 827kB/s] 
 11%|█         | 592k/5.26M [00:00<00:06, 722kB/s]
 85%|████████▍ | 4.45M/5.26M [00:00<00:00, 8.28MB/s]
100%|██████████| 5.26M/5.26M [00:01<00:00, 2.85MB/s]


In [8]:
# Esperar unos segundos a que Kaggle lo puntue antes de correr esto
!kaggle competitions submissions -c playground-series-s6e8

     ref  fileName                         date                        description                                                                     status                     publicScore  privateScore  
--------  -------------------------------  --------------------------  ------------------------------------------------------------------------------  -------------------------  -----------  ------------  
55702279  6_xgboost_optuna_submission.csv  2026-08-23 00:33:22.107000  XGBoost + Optuna (25 trials) - CV AUC=0.96507 - verificado vs semillas y folds  SubmissionStatus.PENDING                              
55587821  6_ensamble_submission.csv        2026-08-17 23:56:17.433000  Ensamble tuneado+lgbm+xgb - OOF AUC=0.96519                                     SubmissionStatus.COMPLETE  0.96654                    
55537898  5_red_neuronal_submission.csv    2026-08-15 23:32:30.953000  Red neuronal Keras - experimento de diversidad - CV AUC=0.94092                 SubmissionStatus.COMPLETE

## 5. Selección para el Scoring Privado

**Esto no se puede hacer por la API**: hay que entrar a la
[página de submissions](https://www.kaggle.com/competitions/playground-series-s6e8/submissions) y
marcar a mano las que cuentan. Al cerrar la competencia **sólo puntúan las seleccionadas**, no la
mejor pública.

La recomendación, basada en CV y no en el leaderboard público:

1. **`6_xgboost_optuna`** — el mejor por validación cruzada, con la ventaja verificada por tres vías
   independientes.
2. **`3_xgboost`** — como segunda. Es la menos correlacionada de las alternativas disponibles y tiene
   score público confirmado, así que funciona como red de seguridad.

Una advertencia sobre la segunda ranura: las tres candidatas de árboles tienen correlaciones de rangos
por encima de 0,999 entre sí, así que **no dan diversificación real**. Elegir dos de ellas es casi
elegir la misma apuesta dos veces. Es una limitación de lo que produjo el episodio, no un descuido de
la selección.

## 6. Cierre del Episodio

| Notebook | Modelo | CV AUC | LB público |
|---|---|---|---|
| 2 | Regresión logística | 0,91687 | 0,91909 |
| 3 | XGBoost | 0,96478 | 0,96634 |
| 4 | LightGBM | 0,96422 | — |
| 4 | Blend LGBM + XGB | 0,96494 | 0,96634 |
| 5 | Red neuronal (Keras) | 0,94092 | 0,94270 |
| 6 | **XGBoost + Optuna** | **0,96507** | *este envío* |

**Lo que funcionó:** los modelos de árboles, y el tuning de hiperparámetros — que aportó +0,0003, una
mejora chica en términos absolutos pero del mismo orden que la brecha entre los puestos 1 y 50 del
leaderboard.

**Lo que no:** cambiar de familia de modelo. Una red neuronal y un foundation model tabular (TabICL)
quedaron ambos ~0,024 por debajo, pese a ser genuinamente más diversos que los GBMs entre sí. La
diversidad resultó condición necesaria pero no suficiente.

**Lo que más se aprendió no fue de modelado sino de medición**: durante buena parte del episodio
descarté mejoras reales por compararlas contra el desvío equivocado. `7_ruido_y_significancia.ipynb`
documenta el error y su corrección, y es probablemente el notebook más útil de los ocho.